EMG Gesture Classification — Subject-Invariant Cross-Validation
=========================================================================
Tests how well the model generalises to unseen subjects (subject invariability).


In [ ]:
import sys
sys.path.append('/kaggle/input/python-module')


In [ ]:
!pip install mantis-tsfm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 13.4 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0


In [ ]:
import os
import copy
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from collections import Counter
from sklearn.model_selection import KFold


In [4]:
def _to_list(value):
    """Accept a scalar or list and always return a list."""
    return list(value) if isinstance(value, (list, tuple, np.ndarray)) else [value]
 
 
def Data_separation(filename, end_point, segment_length, initializer=True):
    """Load a TSV EMG file and return per-gesture slices (classes 1-6)."""
    Data = pd.read_csv(filename, sep="\t").drop(columns="time")
    cut_point   = end_point
    start_point = 0 if initializer else cut_point - segment_length
    return [
        Data[Data['class'] == cls].iloc[start_point:cut_point]
        for cls in range(1, 7)
    ]
 
 
def load_data(filenames, end_point, segment_length, initializer=True):
    """Load and concatenate gesture data from multiple files."""
    gesture_lists = [[] for _ in range(6)]
    for file in filenames:
        for i, df in enumerate(
            Data_separation(file, end_point, segment_length, initializer=initializer)
        ):
            gesture_lists[i].append(df)
    gestures = [pd.concat(lst, ignore_index=True) for lst in gesture_lists]
    return pd.concat(gestures, ignore_index=True)
 
 
def extract_segments(data, segment_length=512, step_size=512):
    """Slide a window and extract fixed-length labelled segments."""
    data    = pd.DataFrame(data)
    signals = data.iloc[:, :-1].values
    labels  = data.iloc[:, -1].values.astype(int)
 
    segments, segment_labels = [], []
    for start in range(0, len(signals) - segment_length + 1, step_size):
        end = start + segment_length
        segments.append(signals[start:end, :])
        segment_labels.append(Counter(labels[start:end]).most_common(1)[0][0])
 
    segments       = np.array(segments)
    segment_labels = np.array(segment_labels) - 1   # zero-index
    return segments, segment_labels
 
 
def resize_to_512(X):
    """Linearly interpolate each segment's time axis to length 512."""
    X_tensor = torch.tensor(X, dtype=torch.float)
    X_scaled = F.interpolate(X_tensor, size=512, mode='linear', align_corners=False)
    return X_scaled.numpy()
 
 
def _load_subject_data(filenames, subject_id):
    """
    Load the two files belonging to one subject.
    Convention: subject s → files at index [2*s, 2*s+1].
    """
    file_indices = [2 * subject_id, 2 * subject_id + 1]
    subject_files = [filenames[i] for i in file_indices]
    return load_data(subject_files, end_point=2000, segment_length=2000,
                     initializer=True)
 
 
def _build_split(filenames, subject_ids, seg_len, step_size, subject_cache):
    """
    Load (with caching), segment, transpose, and resize data for a set of subjects.
 
    Parameters
    ----------
    filenames     : list of all file paths
    subject_ids   : array of subject indices to include
    seg_len       : segment length
    step_size     : step size (already converted from overlap)
    subject_cache : dict — maps subject_id → raw DataFrame (populated here)
 
    Returns
    -------
    X : np.ndarray  (N, channels, 512)
    y : np.ndarray  (N,)
    """
    raw_frames = []
    for s in subject_ids:
        if s not in subject_cache:
            subject_cache[s] = _load_subject_data(filenames, s)
        raw_frames.append(subject_cache[s])
 
    raw = pd.concat(raw_frames, ignore_index=True)
    X, y = extract_segments(raw, segment_length=seg_len, step_size=step_size)
    X = X.transpose(0, 2, 1)         # (N, channels, time)
    X = resize_to_512(X)
    return X, y
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Main cross-validation pipeline
# ─────────────────────────────────────────────────────────────────────────────
 
def run_cv_pipeline(
    data_dir:       str        = None,
    raw_data:       dict       = None,
    n_subjects:     int        = 36,
    n_splits                   = 6,
    shuffle:        bool       = True,
    tuning_type:    str        = "head",
    segment_length             = 512,
    overlap                    = 0,
    learning_rate              = 2e-4,
    num_epochs                 = 55,
    random_state:   int        = 42,
) -> dict:
    """
    Subject-invariant K-fold cross-validation pipeline using Mantis-8M.
 
    In each fold a distinct group of subjects is held out as the test set,
    and the model is trained on the remaining subjects. This tests how well
    the model generalises to completely unseen subjects.
 
    Input modes (exactly one must be provided)
    -------------------------------------------
    data_dir : str, optional
        Root directory walked to find all `.txt` TSV files.
    raw_data : dict, optional
        Dict mapping subject_id (int, 0-indexed) → pre-loaded DataFrame.
        Last column must be the class label (1-indexed).
        Skips file discovery and loading entirely.
 
    Parameters
    ----------
    n_subjects : int
        Total number of subjects in the dataset (default 36).
    n_splits : int or list[int]
        Number of CV folds, e.g. 6 or [4, 6]. Each value triggers a full
        independent sweep.
    shuffle : bool
        Whether to shuffle subjects before splitting into folds.
        True is recommended to avoid ordering bias.
    tuning_type : str
        Fine-tuning strategy: ``'head'`` or ``'full'``.
    segment_length : int or list[int]
        Samples per window, e.g. 512 or [256, 512].
    overlap : int or list[int]
        Samples shared between consecutive windows.
        overlap=0 → non-overlapping (step = segment_length).
        Converted internally: step_size = segment_length - overlap.
    learning_rate : float or list[float]
        AdamW learning rate(s).
    num_epochs : int or list[int]
        Training epochs per fold, e.g. 55 or [50, 100].
    random_state : int
        Seed for subject shuffling and reproducibility.
 
    Returns
    -------
    dict
        ``cv_summary``   – dict keyed by (n_splits, seg_len, overlap, lr, epochs)
                           → list of per-fold accuracies.
        ``mean_summary`` – same keys → mean accuracy across folds.
        ``best``         – combo + fold results with highest mean accuracy.
        ``worst``        – combo + fold results with lowest mean accuracy.
        ``all_results``  – same keys → full per-fold detail dicts.
    """
 
    # ── Guard: exactly one input mode ────────────────────────────────────────
    if data_dir is None and raw_data is None:
        raise ValueError("Provide either 'data_dir' (path) or 'raw_data' (dict).")
    if data_dir is not None and raw_data is not None:
        raise ValueError("Provide either 'data_dir' or 'raw_data', not both.")
 
    # ── Normalise inputs to lists ─────────────────────────────────────────────
    split_counts   = _to_list(n_splits)
    segment_lengths = _to_list(segment_length)
    overlaps        = _to_list(overlap)
    learning_rates  = _to_list(learning_rate)
    epoch_counts    = _to_list(num_epochs)
 
    # Validate overlap < segment_length for every combination
    for seg_len in segment_lengths:
        for ovlp in overlaps:
            if ovlp >= seg_len:
                raise ValueError(
                    f"overlap={ovlp} must be less than segment_length={seg_len}. "
                    f"Maximum allowed is segment_length-1 = {seg_len - 1}."
                )
 
    n_combos = (len(split_counts) * len(segment_lengths) * len(overlaps)
                * len(learning_rates) * len(epoch_counts))
    multi    = n_combos > 1
 
    # ── STEP 1 & 2: Obtain file list or validate raw_data ────────────────────
    print("=" * 65)
    if raw_data is not None:
        print("STEP 1 & 2: Using provided raw_data dict  [file loading skipped]")
        print(f"  Subjects provided: {sorted(raw_data.keys())}")
        filenames     = None
        subject_cache = dict(raw_data)   # already loaded
    else:
        print("STEP 1: Discovering data files")
        filenames = sorted([                # sorted → consistent subject ordering
            os.path.join(root, f)
            for root, _, files in os.walk(data_dir)
            for f in files if f.endswith(".txt")
        ])
        print(f"  Found {len(filenames)} file(s)  ({len(filenames)//2} subjects assumed)")
        print("\nSTEP 2: Files will be loaded per-subject on demand  [cached after first load]")
        subject_cache = {}   # populated lazily during fold construction
 
    # ── STEP 3: Load pretrained Mantis-8M — done ONCE ────────────────────────
    print("\nSTEP 3: Loading pretrained Mantis-8M  [done once]")
    from mantis.architecture import Mantis8M
    from mantis.trainer import MantisTrainer
 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}")
    base_network = Mantis8M(device=device).from_pretrained("paris-noah/Mantis-8M")
 
    # ── STEP 4: Sweep all hyperparameter combos ───────────────────────────────
    subjects   = np.arange(n_subjects)
    cv_summary   = {}   # combo_key → list of per-fold accuracies
    mean_summary = {}   # combo_key → mean accuracy
    all_results  = {}   # combo_key → list of per-fold detail dicts
 
    combos = list(itertools.product(
        split_counts, segment_lengths, overlaps, learning_rates, epoch_counts
    ))
 
    for combo_idx, (n_spl, seg_len, ovlp, lr, epochs) in enumerate(combos, 1):
        combo_key = (n_spl, seg_len, ovlp, lr, epochs)

        if 0 < overlap < 1:
            step_size = int(seg_len * overlap)
        else:
            step_size = seg_len - ovlp   # core conversion: overlap → step
        print(f"Step size: {step_size}")
 
        print(f"\n{'=' * 65}")
        print(f"COMBO {combo_idx}/{len(combos)} │ folds={n_spl}  seg_len={seg_len}  "
              f"overlap={ovlp}  lr={lr:.2e}  epochs={epochs}")
        print(f"{'=' * 65}")
 
        kf = KFold(n_splits=n_spl, shuffle=shuffle,
                   random_state=random_state if shuffle else None)
 
        fold_accuracies = []
        fold_details    = []
 
        for fold_idx, (train_subj_idx, test_subj_idx) in enumerate(kf.split(subjects), 1):
            train_subjects = subjects[train_subj_idx]
            test_subjects  = subjects[test_subj_idx]
 
            print(f"\n  {'─' * 59}")
            print(f"  Fold {fold_idx}/{n_spl}")
            print(f"  Train subjects: {train_subjects.tolist()}")
            print(f"  Test  subjects: {test_subjects.tolist()}")
 
            # Build train and test sets (subject cache avoids re-reading files)
            X_train, y_train = _build_split(
                filenames, train_subjects, seg_len, step_size, subject_cache
            )
            X_test, y_test = _build_split(
                filenames, test_subjects, seg_len, step_size, subject_cache
            )
            print(f"  Train: {X_train.shape}   Test: {X_test.shape}")
 
            # Fresh pretrained weights for every fold — prevents leakage
            network = copy.deepcopy(base_network)
            model   = MantisTrainer(device=device, network=network)
 
            def make_optimizer(lr_val):
                def init_optimizer(params):
                    return torch.optim.AdamW(
                        params, lr=lr_val, betas=(0.9, 0.999), weight_decay=0.05
                    )
                return init_optimizer
 
            print(f"  Fine-tuning  type='{tuning_type}'  epochs={epochs}  lr={lr:.2e}")
            model.fit(
                X_train, y_train,
                num_epochs=epochs,
                fine_tuning_type=tuning_type,
                init_optimizer=make_optimizer(lr),
            )
 
            y_pred   = model.predict(X_test)
            accuracy = float(np.mean(y_test == y_pred))
            print(f"  ✓ Fold {fold_idx} accuracy: {accuracy:.4f}")
 
            fold_accuracies.append(accuracy)
            fold_details.append({
                "fold":          fold_idx,
                "train_subjects": train_subjects.tolist(),
                "test_subjects":  test_subjects.tolist(),
                "accuracy":       accuracy,
                "y_test":         y_test,
                "y_pred":         y_pred,
                "model":          model,
            })
 
        mean_acc = float(np.mean(fold_accuracies))
        std_acc  = float(np.std(fold_accuracies))
        print(f"\n  Combo mean accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
 
        cv_summary[combo_key]   = fold_accuracies
        mean_summary[combo_key] = mean_acc
        all_results[combo_key]  = fold_details
 
    # ── Summary ───────────────────────────────────────────────────────────────
    best_key  = max(mean_summary, key=mean_summary.get)
    worst_key = min(mean_summary, key=mean_summary.get)
    overall_mean = float(np.mean(list(mean_summary.values())))
 
    print("\n" + "=" * 65)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 65)
    if multi:
        print(f"  {'folds':>6}  {'seg_len':>8}  {'overlap':>8}  {'lr':>10}  "
              f"{'epochs':>7}  {'mean_acc':>10}  {'std':>8}")
        print("  " + "-" * 65)
        for (n_spl, seg_len, ovlp, lr, epochs), fold_accs in cv_summary.items():
            m = np.mean(fold_accs)
            s = np.std(fold_accs)
            print(f"  {n_spl:>6}  {seg_len:>8}  {ovlp:>8}  {lr:>10.2e}  "
                  f"{epochs:>7}  {m:>10.4f}  {s:>8.4f}")
        print("  " + "-" * 65)
        print(f"  Overall mean accuracy : {overall_mean:.4f}")
        bk = best_key
        wk = worst_key
        print(f"  Best  combo : folds={bk[0]}  seg_len={bk[1]}  overlap={bk[2]}  "
              f"lr={bk[3]:.2e}  epochs={bk[4]}  → {mean_summary[bk]:.4f}")
        print(f"  Worst combo : folds={wk[0]}  seg_len={wk[1]}  overlap={wk[2]}  "
              f"lr={wk[3]:.2e}  epochs={wk[4]}  → {mean_summary[wk]:.4f}")
    else:
        n_spl, seg_len, ovlp, lr, epochs = best_key
        fold_accs = cv_summary[best_key]
        print(f"  folds={n_spl}  seg_len={seg_len}  overlap={ovlp}  "
              f"lr={lr:.2e}  epochs={epochs}")
        print(f"  Per-fold accuracies : {[round(a, 4) for a in fold_accs]}")
        print(f"  Mean ± Std          : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
    print("=" * 65)
 
    return {
        "cv_summary":   cv_summary,
        "mean_summary": mean_summary,
        "all_results":  all_results,
        "best":  {"combo": best_key,  "mean_accuracy": mean_summary[best_key],
                  "fold_accuracies": cv_summary[best_key]},
        "worst": {"combo": worst_key, "mean_accuracy": mean_summary[worst_key],
                  "fold_accuracies": cv_summary[worst_key]},
        "overall_mean": overall_mean,
    }
 
 

# Example Usage
# ________________

## 'head' fine-tuning

In [ ]:
if __name__ == "__main__":
 
    # ── Single run (mirrors original notebook, with bugs fixed) ──────────────
    results = run_cv_pipeline(
        data_dir       = "/kaggle/input/datasets/rifattanbhir/emg-dataset",
        n_subjects     = 36,
        n_splits       = 6,
        shuffle        = True,
        tuning_type    = "head",
        segment_length = 512,
        overlap        = 256,
        learning_rate  = 2e-4,
        num_epochs     = 125,
        random_state   = 100,
    )
 
    # ── Multi-value sweep ────────────────────────────────────────────────────
    # results = run_cv_pipeline(
    #     data_dir       = "/kaggle/input/datasets/rifattanbhir/emg-dataset",
    #     n_subjects     = 36,
    #     n_splits       = [4, 6],              # 2 values
    #     tuning_type    = "head",
    #     segment_length = [256, 512],          # 2 values
    #     overlap        = [0, 128],            # 2 values
    #     learning_rate  = [1e-4, 2e-4],        # 2 values
    #     num_epochs     = [55, 100],           # 2 values  → 2×2×2×2×2 = 32 combos
    # )
 
    # print(f"\nBest  : {results['best']}")
    # print(f"Worst : {results['worst']}")
    # print(f"Overall mean: {results['overall_mean']:.4f}")
 
    # Extract per-fold accuracies for a specific combo:
    # fold_accs = results["cv_summary"][(6, 512, 0, 2e-4, 55)]
 
    # Access full detail for fold 1 of the best combo:
    # detail = results["all_results"][results["best"]["combo"]][0]


STEP 1: Discovering data files
  Found 72 file(s)  (36 subjects assumed)

STEP 2: Files will be loaded per-subject on demand  [cached after first load]

STEP 3: Loading pretrained Mantis-8M  [done once]
  Device: cuda


config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

Step size: 256

COMBO 1/1 │ folds=6  seg_len=512  overlap=256  lr=2.00e-04  epochs=125

  ───────────────────────────────────────────────────────────
  Fold 1/6
  Train subjects: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34]
  Test  subjects: [0, 7, 12, 19, 32, 35]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=125  lr=2.00e-04


Epoch 124: Train Loss 0.2357: 100%|██████████| 125/125 [00:05<00:00, 22.42it/s]


...

Epoch 124: Train Loss 0.2094: 100%|██████████| 125/125 [00:05<00:00, 23.22it/s]


  ✓ Fold 3 accuracy: 0.7451

  ───────────────────────────────────────────────────────────
  Fold 4/6
  Train subjects: [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 35]
  Test  subjects: [4, 14, 17, 30, 31, 34]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=125  lr=2.00e-04


Epoch 124: Train Loss 0.2556: 100%|██████████| 125/125 [00:05<00:00, 23.71it/s]


  ✓ Fold 4 accuracy: 0.8841

  ───────────────────────────────────────────────────────────
  Fold 5/6
  Train subjects: [0, 1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 26, 28, 29, 30, 31, 32, 33, 34, 35]
  Test  subjects: [2, 10, 20, 21, 25, 27]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=125  lr=2.00e-04


KeyboardInterrupt: 

In [ ]:
seedList = [1, 5, 42, 100, 123]
rows = []

for s in seedList:
    results = run_cv_pipeline(
        data_dir       = "/kaggle/input/datasets/rifattanbhir/emg-dataset",
        n_subjects     = 36,
        n_splits       = 6,
        shuffle        = True,
        tuning_type    = "head",
        segment_length = 512,
        overlap        = 256,
        learning_rate  = 2e-4,
        num_epochs     = 130,
        random_state   = s,
    )

    # Get per-fold accuracies from cv_summary
    # Since it's a single combo, there's one key
    combo_key  = list(results["cv_summary"].keys())[0]
    fold_accs  = results["cv_summary"][combo_key]  # list of 6 accuracies

    row = {"seed": s}
    for i, acc in enumerate(fold_accs, 1):
        row[f"fold_{i}"] = acc
    row["mean"] = float(np.mean(fold_accs))
    row["std"]  = float(np.std(fold_accs))
    rows.append(row)

# Save as CSV
df = pd.DataFrame(rows)
df.to_csv("cv_results_seedwise.csv", index=False)
print(df.to_string(index=False))


STEP 1: Discovering data files
  Found 72 file(s)  (36 subjects assumed)

STEP 2: Files will be loaded per-subject on demand  [cached after first load]

STEP 3: Loading pretrained Mantis-8M  [done once]
  Device: cuda
Step size: 256

COMBO 1/1 │ folds=6  seg_len=512  overlap=256  lr=2.00e-04  epochs=130

  ───────────────────────────────────────────────────────────
  Fold 1/6
  Train subjects: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 20, 21, 22, 23, 24, 25, 26, 27, 29, 31, 32, 33, 35]
  Test  subjects: [3, 17, 19, 28, 30, 34]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.2430: 100%|██████████| 130/130 [00:05<00:00, 23.32it/s]


  ✓ Fold 1 accuracy: 0.8414

  ───────────────────────────────────────────────────────────
  Fold 2/6
  Train subjects: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24, 25, 28, 30, 31, 32, 34, 35]
  Test  subjects: [21, 23, 26, 27, 29, 33]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.1919: 100%|██████████| 130/130 [00:05<00:00, 23.10it/s]


  ✓ Fold 2 accuracy: 0.7219

  ───────────────────────────────────────────────────────────
  Fold 3/6
  Train subjects: [0, 1, 3, 5, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
  Test  subjects: [2, 4, 10, 14, 24, 25]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=130  lr=2.00e-04


...

  ✓ Fold 4 accuracy: 0.7166

  ───────────────────────────────────────────────────────────
  Fold 5/6
  Train subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30, 31, 32, 34, 35]
  Test  subjects: [0, 1, 15, 22, 25, 33]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.2290: 100%|██████████| 130/130 [00:05<00:00, 23.92it/s]


  ✓ Fold 5 accuracy: 0.7986

  ───────────────────────────────────────────────────────────
  Fold 6/6
  Train subjects: [0, 1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 18, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31, 32, 33, 34, 35]
  Test  subjects: [2, 10, 17, 19, 27, 28]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='head'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.2499: 100%|██████████| 130/130 [00:05<00:00, 24.42it/s]


  ✓ Fold 6 accuracy: 0.8806

  Combo mean accuracy : 0.7941 ± 0.0507

CROSS-VALIDATION SUMMARY
  folds=6  seg_len=512  overlap=256  lr=2.00e-04  epochs=130
  Per-fold accuracies : [0.7576, 0.8182, 0.7932, 0.7166, 0.7986, 0.8806]
  Mean ± Std          : 0.7941 ± 0.0507
 seed   fold_1   fold_2   fold_3   fold_4   fold_5   fold_6     mean      std
    1 0.841355 0.721925 0.825312 0.759358 0.750446 0.878788 0.796197 0.055789
    5 0.825312 0.825312 0.718360 0.766488 0.786096 0.843137 0.794118 0.042654
   42 0.839572 0.791444 0.791444 0.866310 0.853832 0.689840 0.805407 0.059115
  100 0.780749 0.761141 0.737968 0.862745 0.725490 0.853832 0.786988 0.053384
  123 0.757576 0.818182 0.793226 0.716578 0.798574 0.880570 0.794118 0.050698


## 'full' fine-tuning

In [ ]:
res = []
rows = []
seedList = [5, 42, 100, 123]
s = 123
if __name__ == "__main__":
# for s in seedList:
    print(f"Starting experiment with seed number: {s}")
    print('#'*65)
    results = run_cv_pipeline(
        data_dir       = "/kaggle/input/datasets/rifattanbhir/emg-dataset",
        n_subjects     = 36,
        n_splits       = 6,
        shuffle        = True,
        tuning_type    = "full",
        segment_length = 512,
        overlap        = 256,
        learning_rate  = 2e-4,
        num_epochs     = 130,
        random_state   = s,
    )
    # Get per-fold accuracies from cv_summary
    # Since it's a single combo, there's one key
    combo_key  = list(results["cv_summary"].keys())[0]
    fold_accs  = results["cv_summary"][combo_key]  # list of 6 accuracies

    row = {"seed": s}
    for i, acc in enumerate(fold_accs, 1):
        row[f"fold_{i}"] = acc
    row["mean"] = float(np.mean(fold_accs))
    row["std"]  = float(np.std(fold_accs))
    rows.append(row)

    # Save as CSV
    df = pd.DataFrame(rows)
    df.to_csv(f"'full'_cv_results_seed{s}.csv", index=False)
    print(df.to_string(index=False))
#     res.append(results['overall_mean'])
# res


Starting experiment with seed number: 123
#################################################################
STEP 1: Discovering data files
  Found 72 file(s)  (36 subjects assumed)

STEP 2: Files will be loaded per-subject on demand  [cached after first load]

STEP 3: Loading pretrained Mantis-8M  [done once]
  Device: cuda


config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

Step size: 256

COMBO 1/1 │ folds=6  seg_len=512  overlap=256  lr=2.00e-04  epochs=130

  ───────────────────────────────────────────────────────────
  Fold 1/6
  Train subjects: [0, 1, 2, 3, 4, 7, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35]
  Test  subjects: [5, 6, 8, 11, 13, 32]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='full'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.0021: 100%|██████████| 130/130 [45:33<00:00, 21.03s/it]


...

  ✓ Fold 4 accuracy: 0.7291

  ───────────────────────────────────────────────────────────
  Fold 5/6
  Train subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30, 31, 32, 34, 35]
  Test  subjects: [0, 1, 15, 22, 25, 33]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='full'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.0023: 100%|██████████| 130/130 [45:55<00:00, 21.20s/it]


  ✓ Fold 5 accuracy: 0.8093

  ───────────────────────────────────────────────────────────
  Fold 6/6
  Train subjects: [0, 1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 18, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31, 32, 33, 34, 35]
  Test  subjects: [2, 10, 17, 19, 27, 28]
  Train: (2811, 8, 512)   Test: (561, 8, 512)
  Fine-tuning  type='full'  epochs=130  lr=2.00e-04


Epoch 129: Train Loss 0.0023: 100%|██████████| 130/130 [45:54<00:00, 21.19s/it]


  ✓ Fold 6 accuracy: 0.8271

  Combo mean accuracy : 0.7805 ± 0.0325

CROSS-VALIDATION SUMMARY
  folds=6  seg_len=512  overlap=256  lr=2.00e-04  epochs=130
  Per-fold accuracies : [0.7594, 0.7897, 0.7683, 0.7291, 0.8093, 0.8271]
  Mean ± Std          : 0.7805 ± 0.0325
 seed   fold_1   fold_2   fold_3   fold_4   fold_5   fold_6     mean     std
  123 0.759358 0.789661 0.768271 0.729055 0.809269 0.827094 0.780452 0.03247


In [ ]:
!zip -r folder.zip /kaggle/working/


  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/'full'_cv_results_seed123.csv (deflated 41%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 71%)


In [ ]:
from IPython.display import FileLink
FileLink(r'folder.zip')


/kaggle/working/folder.zip